# Step 1: Synthetic Data Generation

Generate synthetic NL→SQL training data using GPT-4 as a teacher model.

**What this notebook covers:**
- Loading seed examples and database schemas
- Three generation strategies: seed expansion, self-instruct, evol-instruct
- Quality checking generated examples
- Batch generation with progress tracking
- Saving raw data to `data/raw/`

# ⚠️ IMPORTANT - READ BEFORE RUNNING

**This notebook will REGENERATE DATA and OVERWRITE existing files.**

## Purpose
This is an **educational walkthrough** demonstrating how the data generation pipeline works. It will:
- Generate NEW synthetic data (costs API credits)
- OVERWRITE `data/raw/sql_generation.jsonl` if it exists

## When to Use This Notebook
✅ **LEARNING**: Understanding how data generation works  
✅ **DEVELOPMENT**: Testing new generation strategies  
✅ **FRESH START**: Starting completely from scratch

## When NOT to Use This Notebook
❌ **EVALUATION**: You have already generated data and want to evaluate your trained model  
❌ **PRODUCTION**: You want to use existing curated data  
❌ **COMPARISON**: You want to compare teacher vs student models

## What You Should Run Instead
If you have completed training and want to evaluate your model, run:
- `notebooks/07_comparison_glm.ipynb` (if you have GLM API)
- `notebooks/07_comparison_anthropic.ipynb` (if you have Anthropic API)

See [docs/notebook-guide.md](docs/notebook-guide.md) for complete guidance.

---

# Step 1: Synthetic Data Generation

Generate synthetic NL→SQL training data using GPT-4 as a teacher model.

**What this notebook covers:**
- Loading seed examples and database schemas
- Three generation strategies: seed expansion, self-instruct, evol-instruct
- Quality checking generated examples
- Batch generation with progress tracking
- Saving raw data to `data/raw/`

In [ ]:
import sys
sys.path.insert(0, '..')

import json
from src.llm.client import TeacherClient
from src.generate.few_shot import SeedLoader, FewShotFormatter
from src.generate.strategies import SeedExpansion, SelfInstruct, EvolInstruct
from src.generate.quality import QualityChecker
from src.generate.batch import BatchGenerator

In [ ]:
# Load seed examples
seeds = SeedLoader.load('tasks/sql_generation/seeds.jsonl')
print(f'Loaded {len(seeds)} seed examples')
seeds[0]

In [ ]:
# Load database schemas
with open('tasks/sql_generation/schemas.sql') as f:
    schema = f.read()

# Initialize teacher client
client = TeacherClient()

In [ ]:
# Strategy 1: Seed Expansion — generate variations of existing examples
expander = SeedExpansion(client)
expanded = expander.generate(num_examples=20, schema=schema, seeds=seeds)
print(f'Generated {len(expanded)} examples via seed expansion')
expanded[0]

In [ ]:
# Strategy 2: Self-Instruct — generate entirely new examples from schema
instructor = SelfInstruct(client)
instructed = instructor.generate(num_examples=20, schema=schema, seeds=seeds)
print(f'Generated {len(instructed)} examples via self-instruct')

In [ ]:
# Strategy 3: Evol-Instruct — increase complexity of existing examples
evolver = EvolInstruct(client)
evolved = evolver.generate(num_examples=20, schema=schema, seeds=seeds)
print(f'Generated {len(evolved)} examples via evol-instruct')

In [ ]:
# Quality check a sample
checker = QualityChecker(client)
all_examples = expanded + instructed + evolved
checked = checker.check_batch(all_examples)
print(f'Passed quality check: {len(checked)} / {len(all_examples)}')

In [ ]:
# Full batch generation (use this for production runs)
generator = BatchGenerator(client, config_path='config/tasks/sql_generation.yaml')
results = generator.run(output_path='data/raw/sql_generation.jsonl')
print(f'Total generated: {len(results)}')